# pipeline

> The headless transcription pipeline: VAD analysis → boundary computation → segment cutting → per-segment model-input conversion → transcription, composed over capability workers via the substrate's `JobQueue`.

Stage outputs are threaded into the next stage's inputs **manually** (run job → read result → submit next) because `submit_sequence` cannot pipe outputs to inputs (CR-16); this module is deliberately a real-world consumer of that gap — every workaround here is pass-2 evidence (see `claude-docs/pass-2-evidence.md`).

HITL approval seams use the cheapest viable form (log + optional CLI prompt) per the cores-cluster guard-rails; each seam carries its 5-field HITL-assist annotation in the docstring.

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue, JobStatus

from cjm_transcription_core.models import (
    PipelineConfig,
    RunManifest,
    SegmentRecord,
    SourceResult,
    new_run_id,
)
from cjm_transcription_core.boundaries import compute_segment_boundaries

logger = logging.getLogger(__name__)

In [ ]:
#| export
def field_of(
    result: Any,          # Capability result — dict over the proxy wire, object in-process
    key: str,             # Field name to read
    default: Any = None,  # Fallback when absent
) -> Any:  # The field value or the default
    """Read a field from a dict-or-object capability result.

    Results cross the worker HTTP boundary as JSON dicts but are dataclass
    instances in-process; every consumer in the ecosystem currently
    re-implements this tolerance at each call site (pass-2 evidence: wire-shape
    normalization belongs in a typed layer, not at every consumer).
    """
    if isinstance(result, dict):
        return result.get(key, default)
    return getattr(result, key, default)

In [ ]:
#| export
async def submit_and_wait(
    queue: JobQueue,   # Started job queue
    instance_id: str,  # Capability instance to invoke
    *,
    timeout: Optional[float] = None,  # Seconds to wait; None = no limit
    **kwargs,          # Forwarded to the capability's execute()
) -> Any:  # The completed job's result payload
    """Submit one capability job, wait for it, and return its result (raise on failure)."""
    job_id = await queue.submit(instance_id, **kwargs)
    job = await queue.wait_for_job(job_id, timeout=timeout)
    if job.status != JobStatus.completed:
        raise RuntimeError(f"{instance_id} job {job_id} {job.status}: {job.error}")
    return job.result

In [ ]:
#| export
def normalize_vad_result(
    result: Any,  # MediaAnalysisResult (or proxy dict) from the VAD capability
) -> Tuple[List[Dict[str, float]], float]:  # (sorted speech chunks [{start, end}], reported duration)
    """Normalize a VAD result into sorted speech chunks + the reported duration.

    Duration comes from the result metadata; returns 0.0 when the capability
    did not report one (callers fall back to an ffmpeg probe).
    """
    ranges = field_of(result, "ranges", []) or []
    metadata = field_of(result, "metadata", {}) or {}
    chunks: List[Dict[str, float]] = []
    for r in ranges:
        start = field_of(r, "start", field_of(r, "start_time", 0.0))
        end = field_of(r, "end", field_of(r, "end_time", 0.0))
        chunks.append({"start": float(start), "end": float(end)})
    chunks.sort(key=lambda c: c["start"])
    duration = float(field_of(metadata, "duration", 0.0) or 0.0)
    return chunks, duration

In [ ]:
#| export
async def analyze_vad(
    queue: JobQueue,
    vad_id: str,          # VAD capability instance id
    audio_path: str,      # Audio file to analyze
    force: bool = False,  # Bypass the VAD capability's (path, config_hash) cache
) -> Tuple[List[Dict[str, float]], float]:  # (speech chunks, reported duration)
    """Run VAD analysis on one audio file."""
    result = await submit_and_wait(queue, vad_id, media_path=audio_path, force=force)
    return normalize_vad_result(result)

In [ ]:
#| export
async def probe_duration(
    queue: JobQueue,
    ffmpeg_id: str,   # ffmpeg capability instance id
    audio_path: str,  # Audio file to probe
) -> float:  # Duration in seconds (0.0 when the probe fails to report one)
    """Probe a media file's duration via the ffmpeg capability's `get_info` action."""
    info = await submit_and_wait(queue, ffmpeg_id, action="get_info", file_path=audio_path)
    return float(field_of(info, "duration", 0.0) or 0.0)

In [ ]:
#| export
async def cut_segments(
    queue: JobQueue,
    ffmpeg_id: str,                      # ffmpeg capability instance id
    audio_path: str,                     # Source audio to cut
    boundaries: List[Dict[str, float]],  # [{start, end}, ...] from compute_segment_boundaries
) -> Tuple[List[Dict[str, Any]], str]:  # (per-segment dicts from ffmpeg, batch_key)
    """Cut the source audio at the computed boundaries via ffmpeg `segment_audio`."""
    result = await submit_and_wait(
        queue, ffmpeg_id,
        action="segment_audio", input_path=audio_path, boundaries=boundaries,
    )
    segments = list(field_of(result, "segments", []) or [])
    batch_key = str(field_of(result, "batch_key", "") or "")
    if not segments:
        raise RuntimeError(f"segment_audio produced no segments for {audio_path}: {result!r}")
    return segments, batch_key

In [ ]:
#| export
async def convert_for_model(
    queue: JobQueue,
    ffmpeg_id: str,            # ffmpeg capability instance id
    input_path: str,           # Segment audio file to normalize
    sample_rate: int = 16000,  # Target sample rate
    channels: int = 1,         # Target channel count
) -> str:  # Path to the model-ready WAV
    """Convert one segment to a model-ready WAV via ffmpeg `convert`.

    Audio prep is an upstream ffmpeg concern (Track 12); transcription
    capabilities receive model-ready files. Threaded manually (run → read
    `output_path` → submit next) because `submit_sequence` cannot pipe step
    outputs to step inputs (CR-16).
    """
    result = await submit_and_wait(
        queue, ffmpeg_id,
        action="convert", input_path=input_path,
        output_format="wav", sample_rate=sample_rate, channels=channels,
    )
    wav_path = field_of(result, "output_path")
    if not wav_path or not Path(wav_path).exists():
        raise RuntimeError(f"convert returned no valid output_path for {input_path}: {result!r}")
    return str(wav_path)

In [ ]:
#| export
async def transcribe_segment(
    queue: JobQueue,
    transcriber_id: str,  # Transcription capability instance id
    audio_path: str,      # Model-ready audio file
    *,
    job_id: str,                # Per-call provenance id (keys the capability's DB row)
    source_start_time: float,   # Segment start in source-audio seconds (provenance)
    source_end_time: float,     # Segment end in source-audio seconds (provenance)
    force: bool = False,        # Bypass the (audio_hash, config_hash) cache
) -> Tuple[str, Dict[str, Any]]:  # (transcribed text, transcriber metadata)
    """Transcribe one model-ready segment, passing per-call provenance kwargs.

    `job_id` / `source_*_time` ride the CR-15 identity/provenance kwarg channel;
    `force` is a per-call control flag (CR-15 category 4).
    """
    result = await submit_and_wait(
        queue, transcriber_id,
        audio=audio_path,
        job_id=job_id,
        source_start_time=source_start_time,
        source_end_time=source_end_time,
        force=force,
    )
    text = str(field_of(result, "text", "") or "")
    metadata = field_of(result, "metadata", {}) or {}
    return text, (dict(metadata) if isinstance(metadata, dict) else {})

In [ ]:
#| export
def tier1_segment_checks(
    boundaries: List[Dict[str, float]],  # Computed segment boundaries
    max_segment_duration: float,         # The configured wall-clock cap
    chunk_count: int,                    # VAD speech-chunk count
) -> List[str]:  # Human-readable warnings (empty = all clear)
    """Tier-1 deterministic pre-filters for the boundary-review seam (no AI)."""
    warnings: List[str] = []
    if chunk_count == 0:
        warnings.append("VAD detected NO speech chunks — source may be silent or non-speech")
    for i, b in enumerate(boundaries[:-1]):
        if (b["end"] - b["start"]) > max_segment_duration:
            warnings.append(
                f"non-final segment {i} exceeds max duration: {b['end'] - b['start']:.1f}s"
            )
    if boundaries:
        final = boundaries[-1]
        if (final["end"] - final["start"]) > 2 * max_segment_duration:
            warnings.append(
                f"final segment unusually long ({final['end'] - final['start']:.1f}s) — long trailing silence?"
            )
    return warnings

In [ ]:
#| export
def tier1_transcript_checks(
    segments: List[SegmentRecord],  # Transcribed segments for one source
) -> List[str]:  # Human-readable warnings (empty = all clear)
    """Tier-1 deterministic pre-filters for the transcript-review seam (no AI)."""
    warnings: List[str] = []
    for s in segments:
        if not s.text.strip():
            warnings.append(f"segment {s.index} produced EMPTY text ({s.duration:.1f}s of audio)")
        elif s.duration > 30 and len(s.text) < 20:
            warnings.append(
                f"segment {s.index}: suspiciously short text ({len(s.text)} chars for {s.duration:.1f}s)"
            )
    return warnings

In [ ]:
#| export
def confirm_seam(
    seam: str,                 # Seam label, e.g. "boundary-review"
    summary_lines: List[str],  # What the operator is being asked to accept
    warnings: List[str],       # Tier-1 warnings (logged prominently)
    assume_yes: bool = False,  # Headless mode: accept without prompting
) -> bool:  # True = proceed, False = operator aborted
    """HITL approval seam in its cheapest viable form (log + optional CLI prompt).

    Per-seam capability annotation (HITL-assist methodology, 5 fields):
      1. signal: per-source summaries + Tier-1 warnings
      2. deterministic pre-filter: the tier1_* check functions (no AI)
      3. modality-bridge candidate: spectrogram render for boundary sanity (future Tier 2)
      4. authoritative verifier: re-transcribe-and-compare via a second capability (future Tier 3)
      5. flywheel capture: accept/abort decisions are logged; durable capture is
         a pass-2 seam-contract concern, not solved here

    NOTE: input() blocks the event loop — acceptable because seams sit between
    stages with no jobs in flight; the pass-2 seam contract needs an async shape.
    """
    for line in summary_lines:
        logger.info(f"[{seam}] {line}")
    for w in warnings:
        logger.warning(f"[{seam}] {w}")
    if assume_yes:
        logger.info(f"[{seam}] auto-accepted (assume_yes)")
        return True
    reply = input(f"[{seam}] proceed? [Y/n] ").strip().lower()
    accepted = reply in ("", "y", "yes")
    logger.info(f"[{seam}] {'accepted' if accepted else 'ABORTED'} by operator")
    return accepted

In [ ]:
#| export
async def run_source(
    queue: JobQueue,
    cfg: PipelineConfig,  # Run configuration
    source_path: str,     # Source audio file
    run_id: str,          # Run id (prefixes per-segment job ids)
    source_index: int,    # Position of this source within the run
) -> Optional[SourceResult]:  # None when the operator aborts at a seam
    """Run the full pipeline for one source: VAD → boundaries → cut → convert → transcribe."""
    t0 = time.time()
    logger.info(f"[src {source_index}] {source_path}")

    # 1. VAD analysis (duration falls back to an ffmpeg probe when unreported)
    chunks, duration = await analyze_vad(queue, cfg.vad_plugin, source_path, force=cfg.force)
    if duration <= 0:
        duration = await probe_duration(queue, cfg.ffmpeg_plugin, source_path)
    logger.info(f"[src {source_index}] VAD: {len(chunks)} speech chunks over {duration:.1f}s")

    # 2. Boundaries (pure logic)
    boundaries = compute_segment_boundaries(chunks, cfg.max_segment_duration, duration)

    # 3. HITL seam: boundary review
    if boundaries:
        longest = max(b["end"] - b["start"] for b in boundaries)
        summary = [f"{Path(source_path).name}: {len(boundaries)} segment(s), longest {longest:.1f}s"]
    else:
        summary = [f"{Path(source_path).name}: no segments computed"]
    if not confirm_seam(
        "boundary-review", summary,
        tier1_segment_checks(boundaries, cfg.max_segment_duration, len(chunks)),
        assume_yes=cfg.assume_yes,
    ):
        return None

    # 4. Cut the source at the boundaries
    raw_segments, batch_key = await cut_segments(queue, cfg.ffmpeg_plugin, source_path, boundaries)

    # 5. Per segment: convert → transcribe (manual output→input threading; CR-16)
    records: List[SegmentRecord] = []
    for seg in raw_segments:
        idx = int(field_of(seg, "index", len(records)))
        seg_path = str(field_of(seg, "output_path", ""))
        start = float(field_of(seg, "start", 0.0))
        end = float(field_of(seg, "end", 0.0))
        wav_path = await convert_for_model(
            queue, cfg.ffmpeg_plugin, seg_path,
            sample_rate=cfg.sample_rate, channels=cfg.channels,
        )
        job_id = f"{run_id}_src{source_index}_seg{idx:04d}"
        text, meta = await transcribe_segment(
            queue, cfg.transcriber_plugin, wav_path,
            job_id=job_id, source_start_time=start, source_end_time=end,
            force=cfg.force,
        )
        logger.info(f"[src {source_index}] seg {idx}: {len(text)} chars")
        records.append(SegmentRecord(
            index=idx, start=start, end=end, duration=end - start,
            segment_path=seg_path, model_input_path=wav_path,
            job_id=job_id, text=text, metadata=meta,
        ))

    # 6. HITL seam: transcript review
    total_chars = sum(len(r.text) for r in records)
    if not confirm_seam(
        "transcript-review",
        [f"{Path(source_path).name}: {len(records)} segment(s), {total_chars} chars total"],
        tier1_transcript_checks(records),
        assume_yes=cfg.assume_yes,
    ):
        return None

    logger.info(f"[src {source_index}] done in {time.time() - t0:.1f}s")
    return SourceResult(
        source_path=source_path, duration=duration,
        vad_chunk_count=len(chunks), batch_key=batch_key, segments=records,
    )

In [ ]:
#| export
def collect_plugin_info(
    manager: PluginManager,   # Manager holding the loaded capabilities
    instance_ids: List[str],  # Instance ids to record
) -> Dict[str, Dict[str, Any]]:  # instance_id -> {name, version, db_path}
    """Record capability identity + data-DB pointers for the run manifest (provenance)."""
    info: Dict[str, Dict[str, Any]] = {}
    for iid in instance_ids:
        meta = (getattr(manager, "plugins", {}) or {}).get(iid)
        if meta is None:
            continue
        manifest = getattr(meta, "manifest", {}) or {}
        info[iid] = {
            "name": meta.name,
            "version": getattr(meta, "version", None),
            "db_path": manifest.get("db_path"),
        }
    return info

In [ ]:
#| export
async def run_pipeline(
    manager: PluginManager,  # Manager with the three capabilities loaded
    queue: JobQueue,         # Started job queue
    cfg: PipelineConfig,     # Run configuration
    sources: List[str],      # Source audio paths, in order
    run_id: Optional[str] = None,  # Override run id (default: generated)
) -> RunManifest:  # Manifest of everything the run produced
    """Run the transcription pipeline over the given sources, in order.

    An operator abort at any seam stops the run; the manifest holds the sources
    completed so far (capability-side caches make re-runs cheap).
    """
    run_id = run_id or new_run_id()
    manifest = RunManifest(
        run_id=run_id,
        created_at=time.time(),
        config=cfg.to_dict(),
        plugins=collect_plugin_info(
            manager, [cfg.vad_plugin, cfg.ffmpeg_plugin, cfg.transcriber_plugin]
        ),
    )
    for i, src in enumerate(sources):
        result = await run_source(queue, cfg, str(src), run_id, i)
        if result is None:
            logger.warning(
                f"run {run_id}: aborted at source {i} ({src}); manifest holds {i} source(s)"
            )
            break
        manifest.sources.append(result)
    return manifest

In [ ]:
# Pure-logic smoke checks (no plugins involved)

# field_of tolerates both shapes
assert field_of({"a": 1}, "a") == 1
class _Obj:
    a = 2
assert field_of(_Obj(), "a") == 2
assert field_of({"x": 1}, "missing", "d") == "d"

# normalize_vad_result handles dict-shaped results + both range key spellings
chunks, dur = normalize_vad_result({
    "ranges": [{"start": 5.0, "end": 9.0}, {"start_time": 0.5, "end_time": 2.0}],
    "metadata": {"duration": 28.0},
})
assert chunks == [{"start": 0.5, "end": 2.0}, {"start": 5.0, "end": 9.0}], chunks
assert dur == 28.0

# tier-1 checks fire on the right shapes
assert tier1_segment_checks([], 300.0, 0) != []
assert tier1_segment_checks([{"start": 0.0, "end": 28.0}], 300.0, 3) == []
assert tier1_transcript_checks([SegmentRecord(0, 0.0, 40.0, 40.0, "a", "b", "j", "")]) != []
assert tier1_transcript_checks([SegmentRecord(0, 0.0, 28.0, 28.0, "a", "b", "j", "plenty of text here")]) == []

# confirm_seam headless path
assert confirm_seam("boundary-review", ["x"], [], assume_yes=True) is True
print("pipeline pure-logic checks OK")

pipeline pure-logic checks OK
